# 07 · Quantise, then measure what it costs

**The decision this notebook supports:** what does serving this actually cost, and what quality am I trading for that?

Almost nobody measures this, which is why it is worth doing. This notebook uses MLX for 4-bit quantisation, so **it runs on Apple Silicon only** (install the MLX lines in `requirements-optional.txt`). On NVIDIA hardware the equivalent tools are TensorRT-LLM, vLLM with AWQ/GPTQ weights, or llama.cpp.

Three questions, in order:

1. How much smaller and faster does 4-bit quantisation make the model?
2. How much quality does that cost, measured on **the same frozen evaluation**?
3. What is the **cost per accepted answer**, counting the failures?

That third number is the one an executive understands.

In [ ]:
# Connect this notebook to the lab's saved paths, data, scorer, and configuration.
# sys controls Python's import search path; json reads/writes JSON; warnings controls warning messages.
# Putting the project root on sys.path lets `from src import lab` find our own lab module.
# The relative '..' path assumes a suitable working directory; importing lab does not load an LLM.
# CFG is a nested dictionary: CFG['model']['name'] looks inside 'model', then reads 'name'.
# filterwarnings('ignore') hides warnings, not exceptions; useful loader advice may therefore be hidden.

import sys, json, warnings
sys.path.insert(0, "..")          # project root, so `from src import lab` works
warnings.filterwarnings("ignore") # hide library deprecation noise
from src import lab
CFG = lab.cfg()
print("lab ready |", CFG["model"]["name"])


## What quantisation actually does

Weights are stored as 32-bit floats by default. Quantisation stores them in fewer bits, typically 4, by grouping weights and recording a scale and an offset per group.

The arithmetic is simple and worth being able to do aloud: 4 bits instead of 32 is **one eighth the memory**, plus a small overhead for the scales. Less memory means less data moved per token, and for generation, memory bandwidth is usually the bottleneck rather than compute. **That is why quantisation makes generation faster, not just smaller.**

In [ ]:
# Estimate storage for the same 494 million weights at different bit widths.
# Underscores in 494_000_000 only improve readability. bits/8 converts bits to bytes;
# dividing by 1024**3 converts bytes to GiB, although the printed label says GB.
# This is weight storage, not total running memory: activations, caches and runtime overhead are extra.

P = 494_000_000
# Each tuple supplies a numeric width and a display name. 32/bits is the ideal weight-only ratio.
# f-string :9s reserves a text column; :5.2f formats a number with two decimal places.
for bits, label in [(32,"float32"), (16,"float16"), (8,"int8"), (4,"int4")]:
    gb = P*bits/8/1024**3
    print(f"{label:9s} {gb:5.2f} GB   {32/bits:.0f}x smaller than fp32")
print("\nPlus roughly 0.5 bits per weight of scale/zero-point overhead at group size 64.")


## Convert with MLX

MLX is Apple's native framework and handles the conversion. This writes a new model directory; it does not touch the original.

In [ ]:
# Convert the configured base checkpoint to an MLX 4-bit checkpoint in a separate output folder.
# subprocess launches a command; sys.executable uses this kernel's Python; pathlib.Path handles paths.
# MLX is an Apple-Silicon tensor/runtime ecosystem, and mlx_lm provides model conversion and inference.
# This existing command converts the base path, not the SFT/DPO adapter trained in earlier notebooks.

import subprocess, sys
from pathlib import Path
OUT = lab.ROOT/"models"/"qwen05b-4bit"
OUT.parent.mkdir(exist_ok=True)

# The list contains executable arguments, not a shell command string. -m runs a Python module;
# -q requests quantization and --q-bits 4 selects four-bit weights.
cmd = [sys.executable, "-m", "mlx_lm", "convert",
       "--hf-path", CFG["model"]["base_path"], "--mlx-path", str(OUT), "-q", "--q-bits", "4"]
print(" ".join(cmd))
# capture_output collects stdout/stderr in memory; text=True decodes them into strings.
# A nonzero returncode signals failure. The following prints show only the tail of the logs;
# this code reports the failure but does not raise automatically or prove conversion succeeded.
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-1500:]); print(r.stderr[-800:] if r.returncode else "converted")


In [ ]:
# Compare the two folders' actual disk usage, including their tokenizer/configuration files.
# Path.rglob('*') walks descendants; is_file filters out folders; stat().st_size is a byte count.
# The generator expression feeds file sizes to sum without making another list of all sizes.
# This is a disk-size comparison, not a measurement of GPU memory.

def dirsize(p):
    p = Path(p)
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file())/1024**3
# dirsize returns GiB. The ratio assumes the converted directory contains files;
# if conversion failed and q4 is zero, the existing division will raise ZeroDivisionError.
orig = dirsize(CFG["model"]["base_path"]); q4 = dirsize(OUT)
print(f"original : {orig:.2f} GB")
print(f"4-bit    : {q4:.2f} GB")
print(f"ratio    : {orig/q4:.2f}x smaller")


## Measure speed properly

Two numbers matter and people conflate them:

- **Time to first token** is what the user experiences as responsiveness. It is dominated by prompt processing, which is called prefill.
- **Tokens per second** after the first token is throughput during decoding.

A long system prompt hurts the first badly and the second not at all. That distinction is worth knowing when someone proposes stuffing more context in.

In [ ]:
# Load the converted model into MLX, generate one response, and time the complete call.
# mlx_lm.load returns a model and tokenizer; generate handles autoregressive text generation.
# Python's time module supplies wall-clock timestamps. This cell loads weights and runs inference.

from mlx_lm import load, generate
import time
qmodel, qtok = load(str(OUT))

# A default argument max_tokens=80 can be overridden by a caller.
# The helper returns a tuple: generated text, elapsed seconds, and a re-tokenized output count.
def timed(prompt, max_tokens=80):
    msgs = [{"role":"user","content":prompt}]
    text = qtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    # Start timing after preparing the chat template. The duration includes prompt processing
    # and decoding; it is not an isolated per-token GPU-kernel benchmark.
    t0 = time.time()
    out = generate(qmodel, qtok, prompt=text, max_tokens=max_tokens, verbose=False)
    dt = time.time() - t0
    # Re-encoding the returned text estimates output token count; generated control tokens may be absent.
    # No sampling/seed setting is supplied here, so decoding follows this installed mlx_lm version's defaults.
    n = len(qtok.encode(out))
    return out, dt, n

# Unpack the helper's three return values. Tokens divided by seconds gives an approximate throughput.
# The first call may include warm-up work, so a single number is not a stable performance estimate.
out, dt, n = timed("What is an electric scooter? Answer in two sentences.")
print(f"{n} tokens in {dt:.2f}s = {n/dt:.1f} tok/s\n")
print(out)


In [ ]:
# Repeat the same timing helper on three prompts, then summarize with a median.
# statistics is Python's standard library for small numerical summaries; median is the middle sorted value.
# The underscore in `_, dt, n` means the returned answer is intentionally not used in this cell.
# These short prompts measure a small serving sample, not a representative workload benchmark.

import statistics
prompts = ["Explain battery range in one sentence.",
           "What should I check before buying an electric scooter?",
           "Summarise the benefits of fast charging."]
rates = []
for p in prompts:
    _, dt, n = timed(p, max_tokens=60)
    # append adds each observed tokens-per-second rate to the list; the semicolon separates statements.
    # Taking the median of rates differs from dividing total tokens by total elapsed time.
    rates.append(n/dt); print(f"  {n:3d} tok  {dt:5.2f}s  {n/dt:6.1f} tok/s")
print(f"\nmedian: {statistics.median(rates):.1f} tok/s on 4-bit MLX")


## The quality cost, on the same frozen evaluation

Never accept a speed claim without the matching quality number. Run the identical development set through the quantised model.

In [ ]:
# Measure the 4-bit model on development questions with the full source documents supplied.
# tqdm.auto draws a progress bar; existing MLX model/tokenizer and timing helpers are reused.
# join combines sorted Markdown files with blank lines; the fact index supplies scoring labels only.
# This writes a quality/timing report after generating all answers.

from tqdm.auto import tqdm
SYSTEM = ("You are the customer assistant for Meridian Motors. Answer only from "
          "Meridian's official documents. If you do not have the information, say so plainly.")
DOCS = "\n\n".join(p.read_text() for p in sorted(lab.RAW.glob("*.md")))
fidx, dev = lab.fact_index(), lab.eval_set("dev")

# with_docs=True is the default. The conditional expression adds documents when enabled;
# this system prompt/separator is not byte-identical to every earlier notebook's prompt.
def qask(q, with_docs=True):
    sysmsg = SYSTEM + ("\n\n=== DOCUMENTS ===\n" + DOCS if with_docs else "")
    text = qtok.apply_chat_template([{"role":"system","content":sysmsg},
                                     {"role":"user","content":q}],
                                    tokenize=False, add_generation_prompt=True)
    # MLX generates up to 120 tokens. This existing helper does not record EOS/truncation status
    # or explicitly choose a greedy sampler, so interpret comparisons with that limitation.
    return generate(qmodel, qtok, prompt=text, max_tokens=120, verbose=False).strip()


In [ ]:
# Run the document-assisted dev evaluation and save its quality, time, and size measurements.
# qask() was defined above; rows holds scores and raw keeps the answer text for inspection.
# The saved throughput is from the earlier three-prompt timing sample, not this full evaluation loop.


# Keep scored rows and original answer strings separately, and start an end-to-end timer.
# Generation plus scoring/loop overhead contributes to elapsed; it is not pure decoding time.
rows, raw, t0 = [], [], time.time()
for q in tqdm(dev, desc="4-bit eval"):
    a = qask(q["question"]); rows.append(lab.score_answer(a, q, fidx))
    raw.append({"id":q["id"],"answer":a,"answerable":q["answerable"]})
elapsed = time.time()-t0
s = lab.summarise(rows)
lab.save_run("11_quantised_eval", {"summary": s, "answers": raw,
    "elapsed_s": elapsed, "size_gb": q4, "median_tok_s": statistics.median(rates)})
print(json.dumps(s, indent=2)); print(f"\ntotal wall time: {elapsed:.0f}s for {len(dev)} questions")


## Cost per accepted answer

The number to carry into an interview. Its definition matters more than its value:

**Cost per accepted answer = total cost of every attempt, including the ones that failed, divided by the number of answers you would actually serve.**

Counting only successful generations understates the true figure, sometimes by a lot, because failures still consume compute. Adjust the assumptions below to your own situation and re-run.

In [ ]:
# Estimate cost per accepted answer under explicit illustrative assumptions.
# The accepted-count formula combines correct factual answers with appropriate refusals.
# The machine hourly price, hosted token prices, and average token counts below are assumptions,
# not measured provider invoices or current verified prices. No API request is made here.

summ = s
n_total = summ["n_total"]
# The saved rates are rounded proportions; multiplying and round() reconstructs an approximate count.
# A heuristic 'accepted' answer is not automatically safe to serve without checking scorer limitations.
n_accepted = round(summ["accuracy_answerable"]*summ["n_answerable"] +
                   summ["appropriate_refusal_rate"]*summ["n_unanswerable"])

# Local inference has no invoice, but it is not free. Price the machine's time.
MACHINE_COST_PER_HOUR = 0.15   # amortised laptop + power, your assumption
# Convert seconds into hours, then multiply by the assumed hourly machine cost.
# This simple model omits utilization constraints, support costs, and concurrent workloads.
local_cost = (elapsed/3600)*MACHINE_COST_PER_HOUR

# Compare with a hosted frontier model on the same volume
# 1e6 means one million: a per-million-token price becomes a per-token multiplier.
# This hypothetical API calculation reuses the local accepted count; API quality is not measured here.
API_IN, API_OUT = 0.20/1e6, 1.20/1e6      # $/token, a cheap hosted tier
avg_in, avg_out = 1800, 90                 # docs in context are expensive
api_cost = n_total*(avg_in*API_IN + avg_out*API_OUT)

print(f"questions attempted     : {n_total}")
print(f"answers you would serve : {n_accepted}  ({100*n_accepted/n_total:.0f}%)")
print()
# max(n_accepted,1) prevents division by zero but does not make a zero-acceptance system viable.
# The 100,000-answer extrapolation assumes the same acceptance rate and linear unit cost.
print(f"local 4-bit  total ${local_cost:.4f}  -> ${local_cost/max(n_accepted,1):.5f} per accepted answer")
print(f"hosted API   total ${api_cost:.4f}  -> ${api_cost/max(n_accepted,1):.5f} per accepted answer")
print()
print(f"at 100,000 answers/month: local ${local_cost/max(n_accepted,1)*1e5:,.0f} vs API ${api_cost/max(n_accepted,1)*1e5:,.0f}")
# Save the assumptions with the estimate so a later reader can see how the numbers were produced.
lab.save_run("12_cost", {"n_accepted": n_accepted, "n_total": n_total,
    "local_cost_per_accepted": local_cost/max(n_accepted,1),
    "api_cost_per_accepted": api_cost/max(n_accepted,1),
    "assumptions": {"machine_cost_per_hour": MACHINE_COST_PER_HOUR,
                    "api_in_per_token": API_IN, "api_out_per_token": API_OUT,
                    "avg_in_tokens": avg_in, "avg_out_tokens": avg_out}})


## Pruning and distillation, which we cannot run here

Two more compression techniques are worth knowing. Know what they are even though a 0.5B model on a laptop is the wrong place to try them:

- **Pruning** removes weights, or whole attention heads or layers, that contribute least, then briefly retrains to recover. It changes the architecture; quantisation does not.
- **Knowledge distillation** trains a small student model to match a large teacher's output distribution rather than the ground-truth labels. The student learns from the teacher's *uncertainty*, which carries more information than a hard label.

In practice the production recipe is usually prune, then distil to recover quality, then quantise for serving.

**What you should be able to say now:** the memory arithmetic of 4-bit, the difference between time to first token and throughput, your measured quality cost, and a cost per accepted answer with its denominator stated.

**Next:** notebook 08 puts every stage in one table and asks you for a decision.